# Text to interpretable features to predictions

In [1]:
import sys, time
sys.path.insert(0, "../src")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from llm_feature_gen import LLMFeatureTransformer
from llm_feature_gen.providers import LocalProvider

posts = pd.read_csv("data/newsgroups_hardware/posts.csv").query("split == 'model'")
X, y = posts["text"].tolist(), posts["label"].tolist()

posts["label"].value_counts()

label
mac    60
pc     60
Name: count, dtype: int64

In [2]:
transformer = LLMFeatureTransformer(provider=LocalProvider())

start = time.perf_counter()
features = transformer.fit(X, y).transform(X)
elapsed = time.perf_counter() - start

print(f"{len(X)} documents -> {features.shape[1]} features in {elapsed:.1f}s")
features.head()

Features saved to outputs/discovered_text_features.json


120 documents -> 10 features in 186.7s


,primary_os_reference,hardware_bus_architecture,processor_family_mentioned,storage_medium_focus,software_ecosystem,peripheral_interface_standard,display_technology_reference,system_configuration_method,brand_ecosystem_association,technical_troubleshooting_context
0,mac_system_software,mac_nubus_pds,motorola_68k,not given by LLM,not_applicable,mac_appletalk_serial,not_applicable,not_applicable,apple_macintosh_line,not_applicable
1,mac_system_software,mac_nubus_pds,motorola_68k,not_applicable,not_applicable,not given by LLM,mac_color_quickdraw_vram,not_applicable,apple_macintosh_line,not_applicable
2,mac_system_software,mac_nubus_pds,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,apple_macintosh_line,not_applicable
3,mac_system_software,not_applicable,motorola_68k,not_applicable,not_applicable,not_applicable,mac_color_quickdraw_vram,not_applicable,apple_macintosh_line,not_applicable
4,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable


In [3]:
list(transformer.get_feature_names_out())

['primary_os_reference',
 'hardware_bus_architecture',
 'processor_family_mentioned',
 'storage_medium_focus',
 'software_ecosystem',
 'peripheral_interface_standard',
 'display_technology_reference',
 'system_configuration_method',
 'brand_ecosystem_association',
 'technical_troubleshooting_context']

In [4]:
F_train, F_test, y_train, y_test = train_test_split(features, y, test_size=0.25, stratify=y, random_state=0)

encoder = OneHotEncoder(handle_unknown="ignore").fit(F_train)
model = LogisticRegression(max_iter=1000).fit(encoder.transform(F_train), y_train)
predictions = model.predict(encoder.transform(F_test))

print(f"accuracy {accuracy_score(y_test, predictions):.3f} on {len(y_test)} held-out documents")

accuracy 0.900 on 30 held-out documents
